In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
data = pd.read_csv("data/training.csv")

In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  str    
 3   Auction                            72983 non-null  str    
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  str    
 7   Model                              72983 non-null  str    
 8   Trim                               70623 non-null  str    
 9   SubModel                           72975 non-null  str    
 10  Color                              72975 non-null  str    
 11  Transmission                       72974 non-null  str    
 12  W

In [4]:
data['PurchDate'] = pd.to_datetime(data['PurchDate'])

In [5]:
pd.set_option('display.max_columns', None)
data

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,WheelType,VehOdo,Nationality,Size,TopThreeAmericanName,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,AUTO,1.0,Alloy,89046,OTHER ASIAN,MEDIUM,OTHER,8155.0,9829.0,11636.0,13600.0,7451.0,8552.0,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,1.0,Alloy,93593,AMERICAN,LARGE TRUCK,CHRYSLER,6854.0,8383.0,10897.0,12572.0,7456.0,9222.0,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,2.0,Covers,73807,AMERICAN,MEDIUM,CHRYSLER,3202.0,4760.0,6943.0,8457.0,4035.0,5557.0,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,2009-12-07,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,AUTO,1.0,Alloy,65617,AMERICAN,COMPACT,CHRYSLER,1893.0,2675.0,4658.0,5690.0,1844.0,2646.0,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,2009-12-07,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,MANUAL,2.0,Covers,69367,AMERICAN,COMPACT,FORD,3913.0,5054.0,7723.0,8707.0,3247.0,4384.0,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,73010,1,2009-12-02,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,BLACK,AUTO,1.0,Alloy,45234,AMERICAN,MEDIUM,FORD,1996.0,2993.0,2656.0,3732.0,2190.0,3055.0,4836.0,5937.0,NaN,NaN,18111,30212,GA,4200.0,0,993
72979,73011,0,2009-12-02,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,SILVER,AUTO,NaN,NaN,71759,AMERICAN,MEDIUM,GM,6418.0,7325.0,7431.0,8411.0,6785.0,8132.0,10151.0,11652.0,NaN,NaN,18881,30212,GA,6200.0,0,1038
72980,73012,0,2009-12-02,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,SILVER,AUTO,1.0,Alloy,88500,AMERICAN,MEDIUM SUV,CHRYSLER,8545.0,9959.0,9729.0,11256.0,8375.0,9802.0,11831.0,14402.0,NaN,NaN,18111,30212,GA,8200.0,0,1893
72981,73013,0,2009-12-02,ADESA,2006,3,CHEVROLET,IMPALA,LS,4D SEDAN LS,WHITE,AUTO,1.0,Alloy,79554,AMERICAN,LARGE,GM,6420.0,7604.0,7434.0,8712.0,6590.0,7684.0,10099.0,11228.0,NaN,NaN,18881,30212,GA,7000.0,0,1974


In [6]:
float_int_col = data.select_dtypes(include=['float64', 'int64']).columns.tolist()
data[float_int_col].corr()

,RefId,IsBadBuy,VehYear,VehicleAge,WheelTypeID,VehOdo,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VehBCost,IsOnlineSale,WarrantyCost
RefId,1.000000,0.023973,0.010580,-0.025404,0.013110,-0.026553,-0.011568,-0.015704,-0.032208,-0.034324,-0.008742,-0.013015,-0.021616,-0.024369,0.032896,0.131898,-0.011873,-0.048851,-0.023478
IsBadBuy,0.023973,1.000000,-0.158886,0.167164,-0.044620,0.082560,-0.109252,-0.102954,-0.087418,-0.083665,-0.109112,-0.104020,-0.103914,-0.100245,-0.061488,0.005796,-0.099911,-0.003697,0.052319
VehYear,0.010580,-0.158886,1.000000,-0.957957,0.259660,-0.283719,0.581146,0.532927,0.581191,0.539230,0.590820,0.548426,0.598490,0.558669,0.280791,0.068276,0.349965,0.057380,-0.268696
VehicleAge,-0.025404,0.167164,-0.957957,1.000000,-0.250801,0.319794,-0.565690,-0.518810,-0.460854,-0.426382,-0.575607,-0.532505,-0.504062,-0.469359,-0.268269,-0.074857,-0.312329,-0.024771,0.259310
WheelTypeID,0.013110,-0.044620,0.259660,-0.250801,1.000000,-0.213558,-0.095471,-0.129597,-0.072700,-0.102019,-0.088597,-0.121211,-0.074420,-0.102488,0.187199,0.007901,-0.162826,-0.006747,-0.132798
VehOdo,-0.026553,0.082560,-0.283719,0.319794,-0.213558,1.000000,-0.017022,0.024617,0.031306,0.063165,-0.028568,0.012107,0.015811,0.048005,-0.288456,-0.053041,-0.062008,0.029556,0.411718
MMRAcquisitionAuctionAveragePrice,-0.011568,-0.109252,0.581146,-0.565690,-0.095471,-0.017022,1.000000,0.990347,0.910391,0.910187,0.937817,0.932492,0.871755,0.871042,0.107734,0.045694,0.789840,0.038818,-0.050417
MMRAcquisitionAuctionCleanPrice,-0.015704,-0.102954,0.532927,-0.518810,-0.129597,0.024617,0.990347,1.000000,0.902718,0.918242,0.923610,0.932114,0.860716,0.871203,0.062502,0.040675,0.781516,0.038667,-0.020644
MMRAcquisitionRetailAveragePrice,-0.032208,-0.087418,0.581191,-0.460854,-0.072700,0.031306,0.910391,0.902718,1.000000,0.990201,0.851647,0.849407,0.913108,0.906054,0.107516,0.035489,0.746126,0.080627,-0.053444
MMRAcquisitonRetailCleanPrice,-0.034324,-0.083665,0.539230,-0.426382,-0.102019,0.063165,0.910187,0.918242,0.990201,1.000000,0.846914,0.855689,0.903188,0.906810,0.069676,0.031284,0.744966,0.077670,-0.026933


In [7]:
data.drop(columns=['WheelTypeID', 'PRIMEUNIT', 'RefId', 'AUCGUART'], inplace=True)
data.drop(columns=['MMRAcquisitionAuctionAveragePrice',
                   'MMRAcquisitionAuctionCleanPrice',
                   'MMRAcquisitonRetailCleanPrice',
                   'MMRCurrentAuctionAveragePrice',
                   'MMRCurrentAuctionCleanPrice',
                   'MMRCurrentRetailAveragePrice',
                   'MMRCurrentRetailCleanPrice',
                   'MMRAcquisitionAuctionAveragePrice',
                   'MMRAcquisitionAuctionCleanPrice',
                   'MMRAcquisitionRetailAveragePrice',
                   'MMRAcquisitonRetailCleanPrice',
                   'MMRCurrentAuctionAveragePrice',
                   'MMRCurrentAuctionCleanPrice',
                   'MMRCurrentRetailCleanPrice'],
          inplace=True)
mmr_columns = [
    'MMRAcquisitionAuctionAveragePrice',
    'MMRAcquisitionAuctionCleanPrice',
    'MMRAcquisitionRetailAveragePrice',
    'MMRAcquisitonRetailCleanPrice',
    'MMRCurrentAuctionAveragePrice',
    'MMRCurrentAuctionCleanPrice',
    'MMRCurrentRetailAveragePrice',
    'MMRCurrentRetailCleanPrice'
]

columns_for_mode = ['Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'VehOdo', 'Nationality', 'Size', 'TopThreeAmericanName']
for name in columns_for_mode:
    data[name] = data[name].fillna(data[name].mode()[0])

In [8]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   IsBadBuy              72983 non-null  int64         
 1   PurchDate             72983 non-null  datetime64[us]
 2   Auction               72983 non-null  str           
 3   VehYear               72983 non-null  int64         
 4   VehicleAge            72983 non-null  int64         
 5   Make                  72983 non-null  str           
 6   Model                 72983 non-null  str           
 7   Trim                  72983 non-null  str           
 8   SubModel              72983 non-null  str           
 9   Color                 72983 non-null  str           
 10  Transmission          72983 non-null  str           
 11  WheelType             72983 non-null  str           
 12  VehOdo                72983 non-null  int64         
 13  Nationality           72983

### Разделение на train/val/test

In [9]:
X = data.drop(columns='IsBadBuy')
y = data['IsBadBuy'].copy()

In [10]:
X = X.sort_values(by=['PurchDate'])
y = y.loc[X.index]

len_data = y.shape[0]
train_end_date = X['PurchDate'].iloc[len_data // 3]
val_end_date = X['PurchDate'].iloc[len_data * 2 // 3]

In [11]:
train_mask = X['PurchDate'] < train_end_date
val_mask = (X['PurchDate'] >= train_end_date) & (X['PurchDate'] < val_end_date)
test_mask = X['PurchDate'] >= val_end_date


X_train = X[train_mask]
X_val = X[val_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_val = y[val_mask]
y_test = y[test_mask]

### Энкодер признаков

In [12]:
str_col = X.select_dtypes(include=['str']).columns.tolist()
str_col.append('PurchDate')
ord_enc = OrdinalEncoder(
    handle_unknown='use_encoded_value', 
    unknown_value=-1,
    encoded_missing_value=-2
)

X_train[str_col] = ord_enc.fit_transform(X_train[str_col])
X_val[str_col] = ord_enc.transform(X_val[str_col])
X_test[str_col] = ord_enc.transform(X_test[str_col])

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

### Джини

In [14]:
def gini(y_true, y_pred_proba):
    auc = roc_auc_score(y_true, y_pred_proba)
    gini = 2 * auc - 1
    return gini

### Нейросеть

In [15]:
class MLP:
    def __init__(self, hidden_size=32, learning_rate=0.01, activation='sigmoid', optimizer_type='sgd'):
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.optimizer_type = optimizer_type
        
        self.beta1 = 0.9
        self.beta2 = 0.999
        self.epsilon = 1e-8
        self.t = 0

        self.m_W1 = None
        self.v_W1 = None
        self.m_b1 = None
        self.v_b1 = None
        self.m_W2 = None
        self.v_W2 = None
        self.m_b2 = None
        self.v_b2 = None
        
        self.W1 = None
        self.b1 = None
        self.W2 = None
        self.b2 = None

        # Функция активации скрытого слоя
        self._activation_init(activation)

            
    def _activation_init(self, activation):
        if activation == 'sigmoid':
            self.activation = self.activation_sigmoid
            self.activation_derivative = self.sigmoid_derivative
        elif activation == 'arctan':
            self.activation = np.arctan
            self.activation_derivative = lambda x: 1 / (1 + x**2)
        elif activation == 'relu':
            self.activation = self.activation_relu
            self.activation_derivative = self.relu_derivative
        elif activation == 'cos':
            self.activation = np.cos
            self.activation_derivative = lambda x: -np.sin(x)
        
    
    def _weight_and_bias(self, input_size):
        self.W1 = np.random.randn(input_size, self.hidden_size) * 0.1
        self.b1 = np.zeros((1, self.hidden_size))
        self.W2 = np.random.randn(self.hidden_size, 1) * 0.1
        self.b2 = np.zeros((1, 1))
        
        if self.optimizer_type == 'adam':
            self.m_W1 = np.zeros_like(self.W1)
            self.v_W1 = np.zeros_like(self.W1)
            self.m_b1 = np.zeros_like(self.b1)
            self.v_b1 = np.zeros_like(self.b1)
            self.m_W2 = np.zeros_like(self.W2)
            self.v_W2 = np.zeros_like(self.W2)
            self.m_b2 = np.zeros_like(self.b2)
            self.v_b2 = np.zeros_like(self.b2)
        
        
    def error_log(self, y_true, y_pred):
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        
    def activation_sigmoid(self, x): 
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        s = self.activation_sigmoid(x)
        return s * (1 - s)

    # ReLU: f(x) = max(0, x), производная f'(x) = 1 если x > 0, иначе 0
    def activation_relu(self, x):
        return np.maximum(0, x)

    def relu_derivative(self, x):
        return (x > 0).astype(float)

    def forward(self, X):
        # прямой ход
        # X*W1+b1 -> функция активации -> hidden_act ->hidden_act*W2+b2
        
        self.h = X.dot(self.W1) + self.b1
        self.h_a = self.activation(self.h)

        self.o = self.h_a.dot(self.W2) + self.b2
        self.o_a = self.activation_sigmoid(self.o)
        
        proba_1 = self.o_a
        proba_0 = 1 - proba_1

        return np.hstack([proba_0, proba_1])
    
    def backward(self, X, y_true):
        # обратный ход
        batch_size = X.shape[0]
        
        # Градиент по выходному слою
        # производная выходного слоя y_pred - y_true
        do = self.o_a - y_true.reshape(-1, 1)  # (batch, 1)
        # # получаем ошибку на каждую строку
        dW2 = (self.h_a.T.dot(do)) / batch_size
        db2 = np.sum(do, axis=0, keepdims=True) / batch_size
        
        # Градиент по скрытому слою
        dh_a = do.dot(self.W2.T)
        dh = dh_a * self.activation_derivative(self.h)
        dW1 = (X.T.dot(dh)) / batch_size
        db1 = np.sum(dh, axis=0, keepdims=True) / batch_size
        
        self.dW1, self.db1 = dW1, db1
        self.dW2, self.db2 = dW2, db2

    def update_weights_sgd(self):
        self.W1 -= self.learning_rate * self.dW1
        self.b1 -= self.learning_rate * self.db1
        self.W2 -= self.learning_rate * self.dW2
        self.b2 -= self.learning_rate * self.db2

    def update_weights_adam(self):
        self.t += 1

        self.m_W1 = self.beta1 * self.m_W1 + (1 - self.beta1) * self.dW1 # куда двигался до этого
        self.v_W1 = self.beta2 * self.v_W1 + (1 - self.beta2) * (self.dW1 ** 2) # на сколько сильно изменялся
        self.m_b1 = self.beta1 * self.m_b1 + (1 - self.beta1) * self.db1
        self.v_b1 = self.beta2 * self.v_b1 + (1 - self.beta2) * (self.db1 ** 2)
        self.m_W2 = self.beta1 * self.m_W2 + (1 - self.beta1) * self.dW2
        self.v_W2 = self.beta2 * self.v_W2 + (1 - self.beta2) * (self.dW2 ** 2)
        self.m_b2 = self.beta1 * self.m_b2 + (1 - self.beta1) * self.db2
        self.v_b2 = self.beta2 * self.v_b2 + (1 - self.beta2) * (self.db2 ** 2)

        # Корректировка bias в начале
        m_W1_hat = self.m_W1 / (1 - self.beta1 ** self.t)
        v_W1_hat = self.v_W1 / (1 - self.beta2 ** self.t)
        m_b1_hat = self.m_b1 / (1 - self.beta1 ** self.t)
        v_b1_hat = self.v_b1 / (1 - self.beta2 ** self.t)
        m_W2_hat = self.m_W2 / (1 - self.beta1 ** self.t)
        v_W2_hat = self.v_W2 / (1 - self.beta2 ** self.t)
        m_b2_hat = self.m_b2 / (1 - self.beta1 ** self.t)
        v_b2_hat = self.v_b2 / (1 - self.beta2 ** self.t)

        # Обновление весов "w -= lr * m_hat / (sqrt(v_hat) + ε)"
        self.W1 -= self.learning_rate * m_W1_hat / (np.sqrt(v_W1_hat) + self.epsilon)
        self.b1 -= self.learning_rate * m_b1_hat / (np.sqrt(v_b1_hat) + self.epsilon)
        self.W2 -= self.learning_rate * m_W2_hat / (np.sqrt(v_W2_hat) + self.epsilon)
        self.b2 -= self.learning_rate * m_b2_hat / (np.sqrt(v_b2_hat) + self.epsilon)

    def fit(self, X, y, epochs=100, batch_size=32):
        if hasattr(X, 'values'):
            X = X.values
        if hasattr(y, 'values'):
            y = y.values
            
        self._weight_and_bias(X.shape[1])
        
        for epoch in range(epochs):
            indices = np.random.permutation(len(X))
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            # разбиваем на батчи
            for i in range(0, len(X), batch_size):
                X_temp = X_shuffled[i:i + batch_size]
                y_temp = y_shuffled[i:i + batch_size]

                self.forward(X_temp)
                self.backward(X_temp, y_temp)
                if self.optimizer_type == 'adam':
                    self.update_weights_adam()
                elif self.optimizer_type == 'sgd':
                    self.update_weights_sgd()
            
    def predict_proba(self, X):
        if hasattr(X, 'values'):
            X = X.values
            
        return self.forward(X)

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba[:, 1] > 0.5).astype(int)

In [ ]:
network = MLP(hidden_size=100, learning_rate=0.001, activation='sigmoid')
network.fit(X_train, y_train)
y_pred_proba = network.predict_proba(X_val)

y_pred = y_pred_proba[:, 1]
gini_val = gini(y_val, y_pred)

In [ ]:
print("Метрика Джини для моей нейронки: %f" % gini_val)

### Из библиотеки

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), activation='logistic', learning_rate_init=0.001)
mlp.fit(X_train, y_train)
y_pred_proba_mlp = mlp.predict_proba(X_val)

y_pred_mlp = y_pred_proba_mlp[:, 1]
gini_val_mlp = gini(y_val, y_pred_mlp)

In [ ]:
print("Метрика Джини для MLP из sklearn: %f" % gini_val_mlp)

Оценка результата для датасета из библиотеки лучше на ~0,1. Предполжу, что это связано с тем, что по умолчанию в атрибутах заложено среднее значение, которое немного, но улучшает предсказания модели.\
\
Моя же реализация примитивна и проста по сравнению с методом MLPClassifier, но при идентичных задаваемых параметрах совсем не намного отстает от библиотечной версии.

### Проверка функций активации

In [ ]:
options_activation = ['sigmoid', 'relu', 'cos', 'arctan']
gini_best = -np.inf
temp = None

for act in options_activation:
    network = MLP(hidden_size=100, learning_rate=0.001, activation=act)
    network.fit(X_train, y_train)
    y_pred_proba = network.predict_proba(X_val)
    
    y_pred = y_pred_proba[:, 1]
    gini_val = gini(y_val, y_pred)
    print(f"Для функции активации {act}, Джини = {gini_val:.4f}")
    
    if gini_val > gini_best:
        gini_best = gini_val
        temp = act
print()
print("Самый лучший результат у %s с результатом %f" % (temp, gini_best))

Лучшая функция активации для параметров `hidden_size=100, learning_rate=0.001` - ReLu (также хорошо себя проявил arctan, но это моё желание добавить доп метод). На проверочном наборе данных результат более 0,31

### Нейросеть с помощью PyTorch

In [ ]:
class net_pytorch(nn.Module):
    def __init__(self, hidden_size):
        super(net_pytorch, self).__init__()
        self.hidden_size = hidden_size
        self.fc1 = None
        self.fc2 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        if self.fc1 is None:
            input_size = x.shape[1]
            self.fc1 = nn.Linear(input_size, self.hidden_size)
            # для переноса к остальным слоям
            self.fc1 = self.fc1.to(x.device)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = net_pytorch(hidden_size=100)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.001, weight_decay=0.01)

X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_tensor_val = torch.tensor(X_val, dtype=torch.float32)
y_tensor_val = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)

dataset = TensorDataset(X_tensor, y_tensor)
val = TensorDataset(X_tensor_val, y_tensor_val)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(val, batch_size=32, shuffle=False)


for epoch in range(20):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        net_out = net(data)
        loss = criterion(net_out, target)
        loss.backward()
        optimizer.step()
        
        # if batch_idx % 1000 == 0:
        #     print(f'Epoch: {epoch}, Loss: {loss.item():.6f}')

In [ ]:
net.eval()  
all_probs = []
all_targets = []

with torch.no_grad():
    for data, target in test_loader:
        net_out = net(data)
        probs = torch.sigmoid(net_out)
        
        all_probs.append(probs)
        all_targets.append(target)

y_pred_proba = torch.cat(all_probs, dim=0).numpy().flatten()
y_true = torch.cat(all_targets, dim=0).numpy().flatten()

gini_val = gini(y_true, y_pred_proba)
print(f'Для эпохи {epoch} Gini: {gini_val:.4f}')

Из-за большого количества эпох обучение идет долго, для 100 эпох Gini: 0.2998. Моя нейросеть даже немного превзошла по результатам.
В моей нейросети я умножала на 0.1, а PyTorch nn.Linear использует инициализацию Ксавье по умолчанию. Также есть критерий случайности. 

### Тестирование лучшей моделиь

In [ ]:
network = MLP(hidden_size=100, learning_rate=0.001, activation='relu')
network.fit(X_train, y_train)
y_pred_proba_train = network.predict_proba(X_train)
y_pred_proba_val = network.predict_proba(X_val)
y_pred_proba_test = network.predict_proba(X_test)

gini_train = gini(y_train, y_pred_proba_train[:, 1])
gini_val = gini(y_val, y_pred_proba_val[:, 1])
gini_test = gini(y_test, y_pred_proba_test[:, 1])
print(f"Для train Джини = {gini_train:.4f}")
print(f"Для val Джини = {gini_val:.4f}")
print(f"Для test Джини = {gini_test:.4f}")

Наблюдается небольшое переобучение: Джини падает с 0.3837 на 0.3047 при переходе от Train к Val. Модель показывает результаты хуже на данных, которые не знает.
Но на Test значение растёт (0.3452), но всё равно остаётся ниже, чем на Train.

### Бонус

In [ ]:
network = MLP(hidden_size=100, activation='sigmoid', optimizer_type='adam')
network.fit(X_train, y_train)
y_pred_proba = network.predict_proba(X_val)

y_pred_adam = y_pred_proba[:, 1]
gini_val = gini(y_val, y_pred_adam)

In [ ]:
print("Метрика Джини для нейронки c Adam: %f" % gini_val)

Для Адама характерно переобучение

In [ ]:
!git add ml8.ipynb
!git commit -m "The end"
!git push origin develop